<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementario para el libro <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositorio de código: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Capítulo 6 (adaptado): Fine-tuning de GPT-2 para clasificación de emociones

**Tarea 4 — Extensión LLM**

Este notebook replica el *benchmark* del notebook `RNN_GRU_LSTM` usando un LLM:
se toma la arquitectura **GPT-2 small (124 M)** del Capítulo 6 de *LLMs from Scratch*
y se le aplica fine-tuning de clasificación sobre el
[Emotion Dataset for Emotion Recognition Tasks](https://www.kaggle.com/datasets/parulpandey/emotion-dataset).

**Diferencias respecto al notebook original**

| Aspecto | Notebook RNN/GRU/LSTM | Este notebook |
|---|---|---|
| Backbone | Embedding + BiRNN apilable | GPT-2 small 124 M (pretrained) |
| Tokenizador | `tiktoken` GPT-2 (`enc.encode`) | `tiktoken` GPT-2 (`<\|endoftext\|>` padding) |
| Cabeza de clasificación | `nn.Linear(hidden*2, 6)` | `nn.Linear(emb_dim=768, 6)` sobre el último token |
| Capas entrenables | Todos los pesos desde cero | Último transformer block + `final_norm` + cabeza |
| Padding | Ceros, `pack_padded_sequence` | `<\|endoftext\|>` (id 50256), sin empaquetado |

El protocolo experimental es idéntico: **10 repeticiones** con semillas `1000…1009`,
partición fija (archivos del dataset), *best-val-acc checkpoint*, métricas macro.

**M Rivera / adaptación LLM** · mayo 2026

## 0. Verificación de versiones

In [ ]:
# Idéntico al cell de versiones del Capítulo 6 de Raschka
from importlib.metadata import version

pkgs = [
    "matplotlib",   # Graficas
    "numpy",        # Algebra lineal
    "tiktoken",     # Tokenizador GPT-2
    "torch",        # Framework deep learning
    "pandas",       # Carga del dataset
    "kagglehub",    # Descarga del dataset desde Kaggle
    "scipy",        # Pruebas estadisticas
]
for p in pkgs:
    try:
        print(f"{p} version: {version(p)}")
    except Exception:
        print(f"{p}: no instalado — ejecuta: pip install {p}")

## 1. Descarga del dataset (Kaggle)

El dataset contiene mini-mensajes en inglés etiquetados con seis emociones:
**sadness, joy, love, anger, fear, surprise**.  
Proviene de Kaggle: [`parulpandey/emotion-dataset`](https://www.kaggle.com/datasets/parulpandey/emotion-dataset).

Los archivos `train.csv`, `validation.csv` y `test.csv` se usan tal cual;
**no se mezclan ni re-parten** entre experimentos.

In [ ]:
#!pip install tiktoken kagglehub llms_from_scratch

import os, glob
import pandas as pd

import kagglehub

path = kagglehub.dataset_download("parulpandey/emotion-dataset")
print("Dataset path:", path)

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print("Archivos encontrados:", csv_files)

train_df = pd.read_csv([f for f in csv_files if "train"      in f.lower()][0])
val_df   = pd.read_csv([f for f in csv_files if "validation" in f.lower()][0])
test_df  = pd.read_csv([f for f in csv_files if "test"       in f.lower()][0])

print(f"\nTrain:      {len(train_df):>6,} ejemplos")
print(f"Validation: {len(val_df):>6,} ejemplos")
print(f"Test:       {len(test_df):>6,} ejemplos")
print()
print("Distribución de clases (train):")
print(train_df["label"].value_counts().to_string())
print()
train_df.head()

### Etiquetas numéricas → nombres de clase

El dataset codifica las emociones con enteros 0–5.
La correspondencia es: `{0: sadness, 1: joy, 2: love, 3: anger, 4: fear, 5: surprise}`.

In [ ]:
LABEL_NAMES = ["sadness", "joy", "love", "anger", "fear", "surprise"]
NUM_CLASSES  = len(LABEL_NAMES)

# Verificar que las etiquetas numericas coinciden con la descripcion del dataset
print("Etiquetas únicas en train:", sorted(train_df["label"].unique()))

## 2. Tokenizador tiktoken con codificación GPT-2

Seguimos la convención del Capítulo 6 de Raschka:

- `tiktoken.get_encoding("gpt2")` (equivalente a `encoding_for_model("gpt2")`).
- Token de relleno: `<|endoftext|>` (id **50256**) — el mismo usado en el libro.
- Longitud máxima: se fija a la del ejemplo más largo de **entrenamiento**
  (equivalente a `max_length=None` en `SpamDataset`); los ejemplos más cortos
  se rellenan, los más largos se truncan.

In [ ]:
import tiktoken

tokenizer  = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
pad_token  = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

print(f"GPT-2 vocab size : {vocab_size:,}")
print(f"Padding token id : {pad_token}   (token '<|endoftext|>')")

## 3. Dataset de PyTorch

`EmotionDataset` sigue la estructura de `SpamDataset` del Capítulo 6 de Raschka:

- Pre-tokeniza todos los textos en `__init__`.
- `max_length` se determina por el set de entrenamiento y se reutiliza en
  validación y test (con truncado si es necesario).
- El relleno es `<|endoftext|>` — consistente con el pre-entrenamiento de GPT-2.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EmotionDataset(Dataset):
    """Dataset de emociones compatible con el GPT-2 del Capítulo 6 de Raschka.

    Tokeniza los textos con la codificación GPT-2 de tiktoken y rellena/trunca
    a `max_length` tokens usando el token especial '<|endoftext|>' (id=50256).
    """

    def __init__(self, dataframe, tokenizer, max_length=None, pad_token_id=50256):
        self.labels = dataframe["label"].tolist()

        # Pre-tokenizar todos los textos
        self.encoded_texts = [
            tokenizer.encode(str(text)) for text in dataframe["text"]
        ]

        if max_length is None:
            self.max_length = max(len(ids) for ids in self.encoded_texts)
        else:
            self.max_length = max_length
            # Truncar si excede max_length
            self.encoded_texts = [ids[:max_length] for ids in self.encoded_texts]

        # Rellenar con pad_token_id hasta max_length
        self.encoded_texts = [
            ids + [pad_token_id] * (self.max_length - len(ids))
            for ids in self.encoded_texts
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encoded_texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx],         dtype=torch.long),
        )


# ── Crear datasets (max_length fijo por el set de entrenamiento) ─────────────
train_dataset = EmotionDataset(train_df, tokenizer, max_length=None,
                               pad_token_id=pad_token)

val_dataset   = EmotionDataset(val_df,   tokenizer,
                               max_length=train_dataset.max_length,
                               pad_token_id=pad_token)
test_dataset  = EmotionDataset(test_df,  tokenizer,
                               max_length=train_dataset.max_length,
                               pad_token_id=pad_token)

print(f"Longitud máxima de secuencia (train): {train_dataset.max_length} tokens")
print(f"Ejemplos — train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")

In [ ]:
# Verificar dimensiones de un batch de ejemplo (idéntico al Cell 38 del Cap. 6)
batch_size = 16

train_loader_check = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader_check   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader_check  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

for input_batch, target_batch in train_loader_check:
    break

print("Dimensiones de un batch de entrenamiento:")
print("  input_batch :", input_batch.shape)   # (batch, max_length)
print("  target_batch:", target_batch.shape)  # (batch,)
print()
print(f"Batches — train: {len(train_loader_check)} | val: {len(val_loader_check)} | test: {len(test_loader_check)}")

## 4. Cargar GPT-2 preentrenado y añadir cabeza de clasificación

Tomamos literalmente el procedimiento del **Capítulo 6, secciones 6.4 y 6.5**:

1. Cargar los pesos preentrenados de GPT-2 small (124 M) con `download_and_load_gpt2`.
2. Congelar todos los parámetros (`requires_grad = False`).
3. Reemplazar `model.out_head` con `nn.Linear(768, 6)` (6 emociones).
4. Descongelar el **último bloque transformer** y la **`final_norm`** para que
   sean también entrenables, siguiendo la recomendación de Raschka.

La única diferencia con el notebook original del libro es que `num_classes = 6`
en lugar de 2.

In [ ]:
# Importar utilidades del repositorio llms-from-scratch
# Si no tienes el repositorio clonado, instala el paquete:
#   pip install llms_from_scratch
try:
    from llms_from_scratch.ch05 import download_and_load_gpt2, load_weights_into_gpt
    from llms_from_scratch.ch04 import GPTModel
except ImportError:
    # Fallback: asumir que previous_chapters.py esta en el mismo directorio
    from gpt_download import download_and_load_gpt2
    from previous_chapters import GPTModel, load_weights_into_gpt

# Configuracion identica a la del Capitulo 6 de Raschka
CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG = {
    "vocab_size"    : 50257,
    "context_length": 1024,
    "drop_rate"     : 0.0,
    "qkv_bias"      : True,
}

model_configs = {
    "gpt2-small (124M)" : {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)" : {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)"   : {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size  = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

gpt_base = GPTModel(BASE_CONFIG)
load_weights_into_gpt(gpt_base, params)
gpt_base.eval()

print(f"Modelo cargado: {CHOOSE_MODEL}")
print(f"Dimension de embedding: {BASE_CONFIG['emb_dim']}")

In [ ]:
# ── Sección 6.5 del libro: congelar + reemplazar cabeza ─────────────────────

def build_classifier(base_config, num_classes):
    """Construye el clasificador siguiendo exactamente la Seccion 6.5 de Raschka.

    1. Recarga el modelo base desde cero con los pesos preentrenados.
    2. Congela todos los parametros.
    3. Reemplaza out_head con Linear(emb_dim, num_classes).
    4. Descongela el ultimo bloque transformer y final_norm.
    """
    model = GPTModel(base_config)
    load_weights_into_gpt(model, params)

    # Paso 1: congelar todo
    for param in model.parameters():
        param.requires_grad = False

    # Paso 2: reemplazar cabeza de salida
    torch.manual_seed(123)
    model.out_head = torch.nn.Linear(
        in_features  = base_config["emb_dim"],
        out_features = num_classes,
    )

    # Paso 3: descongelar ultimo bloque y norma final (recomendacion de Raschka)
    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True

    return model


# Verificar conteo de parametros entrenables
_tmp = build_classifier(BASE_CONFIG, NUM_CLASSES)
total   = sum(p.numel() for p in _tmp.parameters())
trainable = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Parametros totales   : {total:>12,}")
print(f"Parametros entrenables: {trainable:>12,}  ({100*trainable/total:.2f} %)")
del _tmp

## 5. Funciones de pérdida y exactitud (Sección 6.6 de Raschka)

Las funciones siguientes son **idénticas** a las del Capítulo 6, con el único
cambio de que `logits[:, -1, :]` sigue siendo el token de salida que se
clasifica, ahora proyectado a 6 clases en lugar de 2.

Se añade `evaluate_full` para calcular *precision*, *recall* y *F1-score* macro
con `sklearn`, métricas requeridas por la tarea.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# ── Idénticas a Capítulo 6 ────────────────────────────────────────────────────

def calc_loss_batch(input_batch, target_batch, model, device):
    """Perdida de un batch: usa el ultimo token como en el Cap. 6."""    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]          # (batch, num_classes)
    return torch.nn.functional.cross_entropy(logits, target_batch)


def calc_loss_loader(data_loader, model, device, num_batches=None):
    """Perdida promedio sobre el loader (igual que Cap. 6)."""    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    n_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= n_batches:
            break
        total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
    return total_loss / n_batches


def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    """Accuracy sobre el loader (igual que Cap. 6)."""    model.eval()
    correct, total = 0, 0
    n_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= n_batches:
            break
        input_batch  = input_batch.to(device)
        target_batch = target_batch.to(device)
        with torch.no_grad():
            logits = model(input_batch)[:, -1, :]
        preds   = torch.argmax(logits, dim=-1)
        correct += (preds == target_batch).sum().item()
        total   += target_batch.size(0)
    return correct / total


# ── Extendida para las 4 métricas requeridas ──────────────────────────────────

def evaluate_full(model, data_loader, device):
    """Devuelve accuracy, precision, recall y F1 macro sobre un loader."""    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for input_batch, target_batch in data_loader:
            logits = model(input_batch.to(device))[:, -1, :]
            preds  = torch.argmax(logits, dim=-1)
            y_true.extend(target_batch.numpy())
            y_pred.extend(preds.cpu().numpy())
    acc  = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

## 6. Ciclo de entrenamiento (Sección 6.7 de Raschka)

`train_classifier_simple` es la función del libro adaptada para registrar
también las 4 métricas de test en el mejor *checkpoint* de validación.

In [ ]:
import random


def set_seed(seed):
    """Fija todas las semillas para reproducibilidad."""    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loaders(seed, batch_size=16):
    """Crea loaders con la particion FIJA del dataset; solo shuffle de train varia con la semilla."""    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, generator=g, drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def train_one_run(seed, device, num_epochs=5, lr=5e-5, weight_decay=0.1):
    """Entrena el clasificador GPT-2 con una semilla dada; devuelve metricas de test.

    Sigue el mismo protocolo que train_classifier_simple del Capitulo 6:
    - AdamW con lr=5e-5 y weight_decay=0.1 (hiperparametros del libro).
    - best-val-accuracy checkpoint.
    - Evaluacion final con los pesos del mejor checkpoint.
    """
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    model = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    best_val_acc = -1.0
    best_state   = None

    for epoch in range(1, num_epochs + 1):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

        val_acc = calc_accuracy_loader(val_loader, model, device)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone()
                            for k, v in model.state_dict().items()}

    # Restaurar mejor checkpoint y evaluar en test
    model.load_state_dict(best_state)
    return evaluate_full(model, test_loader, device)

## 7. Selección de dispositivo (GPU / MPS / CPU)

In [ ]:
# Identico al Cell 82 del Capitulo 6 de Raschka (con soporte MPS para Apple Silicon)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    major, minor = map(int, torch.__version__.split(".")[:2])
    device = torch.device("mps") if (major, minor) >= (2, 9) else torch.device("cpu")
else:
    device = torch.device("cpu")

print("Dispositivo de cómputo:", device)

## 8. Exactitud inicial (antes del fine-tuning)

Igual que en la Sección 6.6 del libro, verificamos las métricas del modelo sin
fine-tuning. Esperamos ~16 % de exactitud (azar uniforme sobre 6 clases).

In [ ]:
_model_init = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
_, _val_loader_init, _test_loader_init = make_loaders(seed=1000)

with torch.no_grad():
    init_train_loss = calc_loss_loader(DataLoader(train_dataset, batch_size=16),
                                       _model_init, device, num_batches=5)
    init_val_loss   = calc_loss_loader(_val_loader_init,  _model_init, device, num_batches=5)

init_train_acc = calc_accuracy_loader(DataLoader(train_dataset, batch_size=16),
                                      _model_init, device, num_batches=5)
init_val_acc   = calc_accuracy_loader(_val_loader_init, _model_init, device, num_batches=5)

print(f"Train loss inicial : {init_train_loss:.4f}  |  Train acc inicial : {init_train_acc:.4f}")
print(f"Val   loss inicial : {init_val_loss:.4f}  |  Val   acc inicial : {init_val_acc:.4f}")
print(f"(Referencia azar uniforme 6 clases: acc ≈ 0.1667)")
del _model_init

## 9. Experimento: 10 repeticiones con semillas 1000–1009

**Protocolo experimental** (idéntico al notebook RNN/GRU/LSTM):

- `N_REPEATS = 10`, semillas `1000…1009`.
- Partición **fija** (archivos del dataset); la aleatoriedad proviene sólo de la
  semilla de inicialización de pesos y del orden de los lotes de entrenamiento.
- `EPOCHS = 5` — suficientes para el fine-tuning con pesos preentrenados.
- Se guarda el *checkpoint* con la mejor *val accuracy* y con él se evalúa test.
- Métricas: *accuracy*, *precision*, *recall*, *F1-score* (todas macro sobre 6 clases).

In [ ]:
N_REPEATS = 10
EPOCHS    = 5

results_llm = []   # lista de dicts {accuracy, precision, recall, f1}

print(f"{'Run':>4}  {'Seed':>5}  {'accuracy':>9}  {'precision':>10}  {'recall':>7}  {'f1':>7}")
print("-" * 55)

for r in range(N_REPEATS):
    seed = 1000 + r
    m    = train_one_run(seed=seed, device=device, num_epochs=EPOCHS)
    results_llm.append(m)
    print(f"{r+1:>4}  {seed:>5}  {m['accuracy']:>9.4f}  {m['precision']:>10.4f}  "
          f"{m['recall']:>7.4f}  {m['f1']:>7.4f}")

print()
print("Experimento finalizado.")

## 10. Promedio y varianza de las métricas

Para cada métrica se reporta la **media**, la **varianza muestral** y la
**desviación estándar** sobre las 10 repeticiones, siguiendo el mismo formato
que la Sección 12 del notebook RNN/GRU/LSTM.

In [ ]:
METRICS = ["accuracy", "precision", "recall", "f1"]

arr_llm = {met: np.array([run[met] for run in results_llm]) for met in METRICS}

rows = []
for met in METRICS:
    v = arr_llm[met]
    rows.append({
        "Métrica"   : met,
        "Media"     : v.mean(),
        "Varianza"  : v.var(ddof=1),
        "Desv. Est.": v.std(ddof=1),
        "Mín."      : v.min(),
        "Máx."      : v.max(),
    })

stats_df = pd.DataFrame(rows).set_index("Métrica")
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
print("Estadísticas del GPT-2 fine-tuned (10 repeticiones, partición fija):")
stats_df

## 11. Tabla comparativa: GPT-2 fine-tuned vs. arquitecturas RNN

Se reconstruyen los resultados del notebook RNN/GRU/LSTM (media ± desv. est.)
y se añaden los del GPT-2 fine-tuned para una comparación directa.

> **Nota:** los valores RNN/GRU/LSTM se ingresan manualmente a partir de la
> ejecución previa del notebook original con las mismas semillas `1000…1009`.

In [ ]:
# ── Resultados del notebook RNN/GRU/LSTM (copiar desde la ejecucion previa) ──
# Formato: (media, desv_est) por metrica
rnn_results = {
    "SimpleRNN": {
        "accuracy" : (0.0, 0.0),   # <-- reemplazar con valores reales
        "precision": (0.0, 0.0),
        "recall"   : (0.0, 0.0),
        "f1"       : (0.0, 0.0),
    },
    "GatedRNN (GRU)": {
        "accuracy" : (0.0, 0.0),
        "precision": (0.0, 0.0),
        "recall"   : (0.0, 0.0),
        "f1"       : (0.0, 0.0),
    },
    "LSTM (variante)": {
        "accuracy" : (0.0, 0.0),
        "precision": (0.0, 0.0),
        "recall"   : (0.0, 0.0),
        "f1"       : (0.0, 0.0),
    },
}

# GPT-2 fine-tuned
gpt2_row = {met: (arr_llm[met].mean(), arr_llm[met].std(ddof=1)) for met in METRICS}

# Construir tabla comparativa
all_models = list(rnn_results.keys()) + ["GPT-2 fine-tuned"]
compare_data = {}

for met in METRICS:
    col = {}
    for name in rnn_results:
        mu, sigma = rnn_results[name][met]
        col[name] = f"{mu:.4f} ± {sigma:.4f}"
    mu, sigma = gpt2_row[met]
    col["GPT-2 fine-tuned"] = f"{mu:.4f} ± {sigma:.4f}"
    compare_data[met.capitalize()] = col

compare_df = pd.DataFrame(compare_data)
compare_df.index.name = "Modelo"
print("Tabla comparativa de desempeño (media ± desv. est. | 10 repeticiones):")
compare_df

## 12. Distribución del desempeño del GPT-2 fine-tuned (boxplots)

Las gráficas siguen exactamente el estilo de la Sección 13 del notebook
RNN/GRU/LSTM para facilitar la comparación visual.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
gpt2_color = "#C44E52"   # rojo para distinguir del notebook RNN (azul/naranja/verde)

for ax, met in zip(axes, METRICS):
    data = arr_llm[met]
    bp   = ax.boxplot([data], labels=["GPT-2\nfine-tuned"],
                      patch_artist=True, showmeans=True, widths=0.5)
    bp["boxes"][0].set_facecolor(gpt2_color)
    bp["boxes"][0].set_alpha(0.7)
    bp["medians"][0].set_color("black")

    # Puntos individuales con jitter
    jitter = np.random.default_rng(0).normal(0, 0.03, size=len(data))
    ax.scatter(np.ones(len(data)) + jitter, data,
               color="black", s=15, alpha=0.6, zorder=3)

    ax.set_title(met.capitalize())
    ax.set_ylim(0.0, 1.0)

fig.suptitle("GPT-2 fine-tuned — distribución del desempeño (10 repeticiones)",
             fontsize=13)
plt.tight_layout()
plt.show()

## 13. Curvas de entrenamiento (run de demostración)

Se entrena una corrida adicional (semilla 42) registrando la pérdida y la
exactitud de entrenamiento y validación en cada época, para reproducir el
gráfico de curvas de aprendizaje que aparece en la Sección 6.7 del libro.

In [ ]:
import time

def train_one_run_with_curves(seed, device, num_epochs=5, lr=5e-5, weight_decay=0.1):
    """Igual que train_one_run, pero tambien devuelve las curvas de loss/acc."""    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    model     = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            calc_loss_batch(input_batch, target_batch, model, device).backward()
            optimizer.step()

        train_loss = calc_loss_loader(train_loader, model, device, num_batches=10)
        val_loss   = calc_loss_loader(val_loader,   model, device)
        train_acc  = calc_accuracy_loader(train_loader, model, device, num_batches=10)
        val_acc    = calc_accuracy_loader(val_loader,   model, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone()
                            for k, v in model.state_dict().items()}

        print(f"  Epoch {epoch:02d}/{num_epochs}  "
              f"train loss={train_loss:.4f}  val loss={val_loss:.4f}  "
              f"train acc={train_acc:.4f}  val acc={val_acc:.4f}  "
              f"({time.time()-t0:.1f}s)")

    model.load_state_dict(best_state)
    test_metrics = evaluate_full(model, test_loader, device)
    return history, test_metrics


print("Corrida de demostración (seed=42):")
history_demo, test_demo = train_one_run_with_curves(seed=42, device=device)
print()
print("Métricas de test (demo):")
for k, v in test_demo.items():
    print(f"  {k:10s}: {v:.4f}")

In [ ]:
# ── Curvas de aprendizaje (estilo Capitulo 6 de Raschka) ─────────────────────
epochs_range = range(1, len(history_demo["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Loss
ax1.plot(epochs_range, history_demo["train_loss"], "b-o", label="Train loss")
ax1.plot(epochs_range, history_demo["val_loss"],   "r-o", label="Val loss")
ax1.set_xlabel("Época")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Curva de pérdida — GPT-2 fine-tuned (seed=42)")
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(epochs_range, history_demo["train_acc"], "b-o", label="Train acc")
ax2.plot(epochs_range, history_demo["val_acc"],   "r-o", label="Val acc")
ax2.set_xlabel("Época")
ax2.set_ylabel("Accuracy")
ax2.set_title("Curva de exactitud — GPT-2 fine-tuned (seed=42)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Matriz de confusión (mejor checkpoint de la corrida demo)

Se reproduce la visualización de la Sección 8 del notebook RNN/GRU/LSTM,
ahora para el GPT-2 fine-tuned.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Re-evaluar con el modelo de la corrida demo para obtener predicciones
set_seed(42)
_, _, test_loader_42 = make_loaders(seed=42)

# Re-entrenamiento rapido solo para obtener el modelo demo guardado
_, _test_loader_demo = make_loaders(42)[1], make_loaders(42)[2]

# Evaluar directamente
model_demo = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
# Nota: para reproducir exactamente, usar el best_state guardado dentro de
# train_one_run_with_curves (version extendida). Aqui hacemos inferencia
# con un modelo entrenado con la misma semilla.

set_seed(42)
_, _, test_loader_cm = make_loaders(seed=42)
y_true_cm, y_pred_cm = [], []

# Usar los resultados de la corrida experimental (seed 1000, primer run)
# para la matriz de confusion (corrida 1 del experimento)
set_seed(1000)
_tl, _vl, _tel = make_loaders(1000)
_m = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
_opt = torch.optim.AdamW(_m.parameters(), lr=5e-5, weight_decay=0.1)

best_val, best_st = -1.0, None
for _ep in range(1, EPOCHS + 1):
    _m.train()
    for _ib, _tb in _tl:
        _opt.zero_grad()
        calc_loss_batch(_ib, _tb, _m, device).backward()
        _opt.step()
    _va = calc_accuracy_loader(_vl, _m, device)
    if _va > best_val:
        best_val = _va
        best_st  = {k: v.detach().cpu().clone() for k, v in _m.state_dict().items()}
_m.load_state_dict(best_st)

_m.eval()
with torch.no_grad():
    for _ib, _tb in _tel:
        _logits = _m(_ib.to(device))[:, -1, :]
        _preds  = torch.argmax(_logits, dim=-1)
        y_true_cm.extend(_tb.numpy())
        y_pred_cm.extend(_preds.cpu().numpy())

cm = confusion_matrix(y_true_cm, y_pred_cm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Conteos absolutos
sns.heatmap(cm, annot=True, fmt="d", cmap="turbo",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")
axes[0].set_title("Confusion Matrix — GPT-2 fine-tuned (conteos)")

# Normalizada por fila
cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="turbo",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")
axes[1].set_title("Confusion Matrix — GPT-2 fine-tuned (normalizada)")

plt.tight_layout()
plt.show()

## 15. Boxplots combinados: GPT-2 vs. RNN/GRU/LSTM

Para incluir los modelos RNN en la comparación gráfica se necesitan sus
distribuciones de 10 runs. La celda siguiente muestra cómo construir la figura
una vez que se copian los arrays del notebook RNN/GRU/LSTM.

> **Instrucción:** copiar los valores `arr["SimpleRNN"]["f1"]` etc. del
> notebook original en el diccionario `arr_rnn` de abajo antes de ejecutar.

In [ ]:
# ── Sustituir con los arrays reales del notebook RNN/GRU/LSTM ─────────────────
# Ejemplo con valores ficticios (reemplazar con los reales)
arr_rnn = {
    "SimpleRNN"      : {met: np.zeros(10) for met in METRICS},
    "GatedRNN (GRU)" : {met: np.zeros(10) for met in METRICS},
    "LSTM (variante)": {met: np.zeros(10) for met in METRICS},
}

all_arr = {**arr_rnn, "GPT-2 fine-tuned": arr_llm}
all_model_names = list(all_arr.keys())
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, met in zip(axes, METRICS):
    data = [all_arr[n][met] for n in all_model_names]
    bp   = ax.boxplot(data, labels=all_model_names, patch_artist=True,
                      showmeans=True, widths=0.6)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp["medians"]:
        median.set_color("black")

    for i, vals in enumerate(data, start=1):
        jitter = np.random.default_rng(i).normal(0, 0.05, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals,
                   color="black", s=10, alpha=0.4, zorder=3)

    ax.set_title(met.capitalize())
    ax.set_ylim(0.0, 1.0)
    ax.tick_params(axis="x", rotation=25)

fig.suptitle("Distribución del desempeño: GPT-2 vs. RNN/GRU/LSTM (10 repeticiones)",
             fontsize=13)
plt.tight_layout()
plt.show()

## 16. Análisis de significancia estadística

### ¿Cuál es el mejor modelo? ¿La diferencia es estadísticamente significativa?

Siguiendo la Parte III del notebook RNN/GRU/LSTM se aplica un **t-test
independiente** (`scipy.stats.ttest_ind`) sobre la métrica **F1 macro** de las
10 repeticiones.  Las hipótesis son:

- $H_0$: la diferencia media de F1 entre los dos modelos es cero.
- $H_1$: la diferencia media de F1 es distinta de cero.

Se aplica la **corrección de Bonferroni** para comparaciones múltiples:
$\alpha_{\text{Bonf}} = 0.05 / \binom{4}{2} = 0.0083$.

In [ ]:
from itertools import combinations
from scipy import stats

PRIMARY = "f1"
alpha   = 0.05

# Ranking de todos los modelos por F1 macro
all_arr_f1 = {name: all_arr[name][PRIMARY] for name in all_model_names}
ranking = sorted(all_arr_f1, key=lambda n: all_arr_f1[n].mean(), reverse=True)

print("Ranking por F1 macro (media ± desv. est. | 10 repeticiones):")
for i, name in enumerate(ranking, 1):
    v = all_arr_f1[name]
    print(f"  {i}. {name:<22}  F1 = {v.mean():.6f} ± {v.std(ddof=1):.6f}")

best = ranking[0]
print(f"\n>>> Mejor modelo por F1 macro: {best}")

In [ ]:
pairs   = list(combinations(all_model_names, 2))
n_pairs = len(pairs)
alpha_bonf = alpha / n_pairs

rows = []
for a, b in pairs:
    va, vb   = all_arr_f1[a], all_arr_f1[b]
    t_stat, t_p = stats.ttest_ind(va, vb)
    rows.append({
        "Comparación"        : f"{a} vs {b}",
        "Diff F1 (a-b)"      : va.mean() - vb.mean(),
        "t-stat"             : t_stat,
        "p-value"            : t_p,
        f"Signif. (α={alpha})"         : "Sí" if t_p < alpha      else "No",
        f"Signif. Bonf. (α={alpha_bonf:.4f})": "Sí" if t_p < alpha_bonf else "No",
    })

test_df = pd.DataFrame(rows)
print(f"Corrección de Bonferroni: α_bonf = {alpha:.2f} / {n_pairs} = {alpha_bonf:.4f}")
test_df

In [ ]:
print("=" * 72)
print("CONCLUSIÓN")
print("=" * 72)

best_vs = []
for a, b in pairs:
    if best in (a, b):
        other    = b if a == best else a
        va, vb   = all_arr_f1[best], all_arr_f1[other]
        _, p     = stats.ttest_ind(va, vb)
        sig_raw  = p < alpha
        sig_bonf = p < alpha_bonf
        best_vs.append((other, va.mean() - vb.mean(), p, sig_raw, sig_bonf))

print(f"El modelo con mejor F1 macro promedio es: {best}  "
      f"(F1 = {all_arr_f1[best].mean():.4f}).\n")

for other, dmean, p, sig_raw, sig_bonf in best_vs:
    rel       = "mejor" que if dmean > 0 else "peor que"
    v_raw     = "SÍ"  if sig_raw  else "NO"
    v_bonf    = "SÍ"  if sig_bonf else "NO"
    print(f"  vs {other}:")
    print(f"    Diferencia en F1 = {dmean:+.4f}  |  p = {p:.4f}")
    print(f"    Significativa (α={alpha})       : {v_raw}")
    print(f"    Significativa (α_Bonf={alpha_bonf:.4f}): {v_bonf}")
    print()

if all(sig_bonf for *_, sig_bonf in best_vs):
    print(f"=> {best} es significativamente superior a TODOS los demás modelos (Bonferroni).")
elif any(sig_bonf for *_, sig_bonf in best_vs):
    print(f"=> {best} supera significativamente a algunos modelos, pero no a todos (Bonferroni).")
else:
    print(f"=> {best} tiene la mejor media, pero sin corrección de Bonferroni\n"
          f"   no hay evidencia suficiente de superioridad sobre todos los demás.")

## 17. Función de inferencia

Equivalente a la Sección 7 del notebook RNN/GRU/LSTM.

In [ ]:
def predict_emotion_gpt2(text, model, tokenizer, device,
                         max_length=train_dataset.max_length,
                         pad_token_id=50256):
    """Predice la emocion de un texto usando el GPT-2 fine-tuned."""    model.eval()
    ids = tokenizer.encode(str(text))[:max_length]
    ids = ids + [pad_token_id] * (max_length - len(ids))
    input_tensor = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
        probs  = torch.softmax(logits, dim=-1)
        pred   = torch.argmax(probs, dim=-1).item()

    return LABEL_NAMES[pred], probs.cpu().numpy()[0]


# ── Demo de inferencia (mismo texto que en el notebook original) ──────────────
# Usar el modelo de la ultima corrida del experimento (seed=1009)
model_final = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
_state = train_one_run(seed=1009, device=device, num_epochs=EPOCHS)
# Nota: train_one_run devuelve metricas, no el modelo.
# Para demo de inferencia volvemos a entrenar rapidamente solo un run.
set_seed(1009)
_tl2, _vl2, _ = make_loaders(1009)
model_final = build_classifier(BASE_CONFIG, NUM_CLASSES).to(device)
_opt2 = torch.optim.AdamW(model_final.parameters(), lr=5e-5, weight_decay=0.1)
_bv2, _bs2 = -1.0, None
for _ep in range(1, EPOCHS + 1):
    model_final.train()
    for _ib, _tb in _tl2:
        _opt2.zero_grad()
        calc_loss_batch(_ib, _tb, model_final, device).backward()
        _opt2.step()
    _va2 = calc_accuracy_loader(_vl2, model_final, device)
    if _va2 > _bv2:
        _bv2 = _va2
        _bs2 = {k: v.detach().cpu().clone() for k, v in model_final.state_dict().items()}
model_final.load_state_dict(_bs2)

test_texts = [
    "I am so happy today!",
    "I feel really scared about the results.",
    "This is absolutely outrageous, I can't believe it.",
    "I miss you so much, everything feels empty.",
    "I love spending time with you.",
    "Wow, I never expected that to happen!",
]

print(f"{'Texto':<45}  {'Predicción':<10}  Probabilidades")
print("-" * 80)
for texto in test_texts:
    emocion, probs = predict_emotion_gpt2(texto, model_final, tokenizer, device)
    prob_str = "  ".join(f"{LABEL_NAMES[i]}={probs[i]:.3f}" for i in np.argsort(probs)[::-1][:3])
    print(f"{texto:<45}  {emocion:<10}  {prob_str}")